# Day 1 · Module 3a — Multimodal Fusion (the Late-Fusion Patient Model)

**Driving question:** *You have a clinical chart AND a chest X-ray for each patient. Does gluing both into one model beat using just the better one — and what happens when the X-ray is missing?*

**Objective:** reuse the Module-2 image network as a frozen feature extractor, fuse its embedding with the Module-1-style tabular features, and discover that fusion is a **measured trade-off** — sometimes the fancier model *loses*.

> **Student notebook** · kernel **AImed**. You build the fused scorer yourself; if you fall behind, a CHECKPOINT cell plots the answer so you can read it off.

### Pre-work recap
- **Fusion taxonomy** (Huang et al. 2020): *early* (concatenate raw inputs), *joint* (merge intermediate features and backprop into every extractor), *late* (combine near the end). This module builds **late** fusion.
- This module **eats Module 2's output**: the image CNN you trained becomes a frozen *encoder* here.
- *Arrive-with (poll):* **When would adding a second modality make a model *worse*?** Hold your answer — you will measure it.

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

### Background — what "late fusion" actually is, and why more data can hurt

New to this? Read first.

**The setup.** Each patient has TWO kinds of data: a **tabular** record (age, labs, vitals — Module 1's world, here 8 numbers per patient) and a **chest image** (Module 2's world). We want one model that uses both.

**The frozen encoder — reusing Module 2.** A trained image CNN is really two parts stacked: a long stack of layers that turn a picture into a list of numbers (the **embedding** — here 512 numbers that summarize "what's in this X-ray"), and a final layer that turns those numbers into a yes/no. In **late fusion** we *chop off that final layer*, keep the 512-number embedding, and treat the CNN as a fixed **feature extractor** ("frozen" = we don't retrain it). That is the literal payoff of the build arc: **Module 2's model becomes Module 3a's input.**

**Late fusion = glue, then a small head.** We paste the image embedding (512 numbers) next to the tabular features (8 numbers) to make one 520-number vector per patient, and train a small classifier (a logistic-regression "head") on top. "Late" because each modality is processed separately first and only joined near the *end*. (Contrast: *early* fusion glues raw inputs at the start; *joint* fusion trains everything together — see Stretch.)

**Analogy — a hiring panel.** Tabular features are a sharp recruiter who knows the job (strong signal). The image embedding is a second interviewer who is *new and unsure* (weak, noisy signal). Late fusion = let both vote and average. If the second interviewer is reliable, the panel beats either alone. But **if the second one mostly guesses, averaging their vote IN drags the good recruiter's decision toward noise.** More opinions are not automatically a better decision — and that is the whole module.

**Why a second modality can hurt (concrete mechanisms):** (1) it adds *parameters* the small head can overfit; (2) it adds *noise* if that modality is weak; (3) **missingness** — not every visit has imaging, and *how* you fill the gap (drop the patient / zero the embedding / add a "present?" flag) silently changes the answer.

**Be honest about the data.** The bundled fusion set (`load_multimodal()`) is a **constructed pairing**: 600 rows where an image embedding sits next to a tabular record with a shared label engineered so fusion *can* help. The tabular and image data are NOT from the same real cohort. We use it because it teaches the *mechanics* (frozen encoder -> concat -> head) cleanly; the clinical validity does not transfer. If that bothers you — good, it should. Name it in the discussion.

In [ ]:
# Guided hands-on — these helpers do the heavy lifting; we PRINT the key numbers.
import late_fusion as LF

img, tab, y = LF.load_multimodal()   # img_emb [600,512], tab [600,8], y in {0,1}
print(f'image embedding: {img.shape}   tabular: {tab.shape}   prevalence: {y.mean():.2f}')

# Three models, ONE shared train/test split, compared by AUROC (ranking quality):
aurocs = LF.three_way(img, tab, y)   # -> {'image_only', 'tabular_only', 'fused'}
for name in ('image_only', 'tabular_only', 'fused'):
    print(f'  {name:13s} AUROC = {aurocs[name]:.3f}')

best_single = max(aurocs['image_only'], aurocs['tabular_only'])
print(f"\n  best single modality = {best_single:.3f}   |   fused = {aurocs['fused']:.3f}")
print('  --> fusion BEAT the best single modality' if aurocs['fused'] > best_single
      else '  --> fusion LOST to the best single modality (this is the lesson)')

### Key terms

| Term | Plain meaning |
|---|---|
| **Embedding** | the list of numbers a network produces *before* its final decision layer — a compressed summary of the input (here 512 numbers per X-ray). |
| **Frozen encoder** | a pretrained network used only to *produce embeddings*; its weights are not updated. Module 2's CNN, head removed. |
| **Late fusion** | concatenate per-modality features near the end, then train a small head. (vs *early* = glue raw inputs; *joint* = train everything end-to-end.) |
| **AUROC** | threshold-free ranking quality: chance a random positive outranks a random negative. 0.5 = coin flip. |
| **Missing modality** | a patient lacks one input (e.g. no X-ray). How you handle it — *drop / zero-impute / flag* — changes the model. |

In [ ]:
# VIVID DEMO — is the 'fusion vs best-single' gap real, or just noise?
# We bootstrap-resample the test set to put an UNCERTAINTY BAND on each AUROC, then we
# corrupt the image modality into pure noise and re-fuse, to SEE noise drag fusion down.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from common import metrics as M

idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)

def fit_scores(Xtr, ytr, Xte):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=0))
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]

# 1) Bootstrap CI on the two that matter: tabular-only vs fused.
for name, X in (('tabular_only', tab), ('fused', np.concatenate([img, tab], 1))):
    s = fit_scores(X[tr], y[tr], X[te])
    mean, lo, hi = M.bootstrap_auroc_ci(y[te], s, n_boot=500, seed=0)
    print(f'{name:13s} AUROC {mean:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')
print('--> if those two intervals OVERLAP, the "fusion difference" is within noise.\n')

# 2) Make the second modality WORTHLESS (replace the image embedding with pure noise),
#    then fuse. Watch the strong tabular signal get dragged DOWN by junk it must average in.
rng = np.random.default_rng(0)
junk = rng.standard_normal(img.shape).astype('float32')
tab_alone = M.auroc(y[te], fit_scores(tab[tr], y[tr], tab[te]))
fused_junk = M.auroc(y[te], fit_scores(np.concatenate([junk, tab], 1)[tr], y[tr],
                                       np.concatenate([junk, tab], 1)[te]))
print(f'tabular alone            AUROC {tab_alone:.3f}')
print(f'tabular + 512 noise cols AUROC {fused_junk:.3f}   (fusing in pure noise can only hurt)')

In [ ]:
# YOUR TURN — BUILD the fused model yourself (don't call three_way this time).
# This is the mastery move: late fusion is literally 'concatenate, then score'.
import numpy as np
from common import metrics as M

# Reuse the split tr/te and the fit_scores() helper from the DEMO cell above.
# (a) Late fusion = paste the image embedding next to the tabular features, column-wise.
#     np.concatenate([A, B], axis=?) -> stack columns so each row keeps all its features.
fused_X = np.concatenate([img, tab], axis=...)        # <-- fill the axis (rows? or columns?)
assert fused_X.shape == (600, 520), 'late fusion glues 512 image + 8 tabular = 520 columns per patient'

# (b) Train the head on the TRAIN rows, score the TEST rows, then measure AUROC.
fused_scores = fit_scores(fused_X[tr], y[tr], fused_X[...])   # <-- which rows do you SCORE on?
my_fused_auroc = M.auroc(y[te], fused_scores)

# (c) Compare to the best single modality you printed in the guided cell.
best_single = max(aurocs['image_only'], aurocs['tabular_only'])
print(f'my hand-built fused AUROC = {my_fused_auroc:.3f}   best single = {best_single:.3f}')
print('fusion helped' if my_fused_auroc > best_single else 'fusion did NOT help here')

In [ ]:
# YOUR TURN (part 2) — the missing-modality drill.
# Reality: ~30% of visits have NO image. Three ways to cope; each biases the model differently.
#   strategy='drop'        -> throw away patients with no image
#   strategy='zero_impute' -> set the missing image embedding to all zeros
#   strategy='flag'        -> zero it AND add a 'image-present?' indicator column
missing = {s: LF.missing_modality(img, tab, y, frac=0.30, strategy=s)
           for s in ('drop', 'zero_impute', 'flag')}
for s, v in missing.items():
    print(f'  {s:12s} fused AUROC @30% missing images = {v:.3f}')

# Which policy would YOU ship? (you defend this in the DECISION cell)
best_policy = ...        # <-- one of 'drop' / 'zero_impute' / 'flag'
print('I would ship:', best_policy)

In [ ]:
# Static visual — the three-way comparison as bars, with the best-single reference line.
%matplotlib inline
import matplotlib.pyplot as plt
names = ['image_only', 'tabular_only', 'fused']
vals = [aurocs[n] for n in names]
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, vals, color=['C1', 'C0', 'C2'])
ax.axhline(max(aurocs['image_only'], aurocs['tabular_only']), ls='--', color='0.4',
           label='best single modality')
ax.axhline(0.5, ls=':', color='r', label='coin flip')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}', ha='center')
ax.set(ylim=(0.4, 1.0), ylabel='AUROC', title='Does fusion clear the best single modality?')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
print('Read it: if the green (fused) bar does NOT rise above the dashed line, fusion did not earn its complexity.')

### Your task — make and defend a fusion decision
The numbers are the easy part. Using them, fill the **`DECISION`** dict: would you **ship the fused model** when it does not beat the best single modality, and *why* (name a concrete mechanism: added noise, overfitting, or missingness)? For the 30%-missing reality, which handling policy do you ship, and what does it *cost* (drop = who disappears from the data? zero = what false signal does an all-zero embedding send? flag = what does the extra column let the head learn?). There is no single right answer — what matters is the reasoning.

In [ ]:
# YOUR TURN — defend the call. No auto-grader: argue it in the discussion, cite real numbers.
DECISION = {
    'ship_fused': ...,        # True/False: ship the fused model over the best single modality?
    'why': ...,               # one sentence: the mechanism (added noise / overfit / CIs overlap / ...)
    'missing_policy': ...,    # 'drop' / 'zero_impute' / 'flag' — which you'd deploy at 30% missing
    'policy_cost': ...,       # one sentence: the bias/cost that policy introduces
    'same_patients': ...,     # one sentence: the caveat that the fusion set is a CONSTRUCTED pairing
}
print(DECISION)

### Checkpoint — stuck? Visualize the whole answer

In [ ]:
# CHECKPOINT (non-blocking) — VISUALIZES the answer so a stuck student can read it off.
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from late_fusion import run_report
st = run_report(missing_frac=0.30, log=lambda *a: None)
a = st['aurocs']; miss = st['missing']
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
# LEFT — three-way comparison
namesL = ['image_only', 'tabular_only', 'fused']
bL = axL.bar(namesL, [a[n] for n in namesL], color=['C1', 'C0', 'C2'])
axL.axhline(max(a['image_only'], a['tabular_only']), ls='--', color='0.4', label='best single')
axL.set(ylim=(0.4, 1.0), ylabel='AUROC', title='Three-way: does fused beat best single?')
for b in bL: axL.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}', ha='center')
axL.legend(loc='lower right')
# RIGHT — missing-modality policies
namesR = ['drop', 'zero_impute', 'flag']
bR = axR.bar(namesR, [miss[n] for n in namesR], color='C3')
axR.axhline(a['fused'], ls='--', color='0.4', label='fused, no missing')
axR.set(ylim=(0.4, 1.0), ylabel='fused AUROC', title='Cost of 30% missing images')
for b in bR: axR.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}', ha='center')
axR.legend(loc='lower right')
plt.tight_layout(); plt.show()
print('Left: if green (fused) is below the dashed line, fusion lost. Right: each policy lands at a different AUROC.')

**How to read the checkpoint plot.**

*Left — the three-way comparison (does fusion earn its complexity?).* **Analogy — the hiring panel.** Orange = the unsure new interviewer (image-only), blue = the sharp recruiter (tabular-only), green = their averaged panel vote (fused). The **dashed line is the best single voter.** If the green bar does NOT rise above the dashed line, adding the second voter did not help — and if it sits *below*, the noisy voter actively dragged the decision down. That is the counterintuitive headline: **the fancier multimodal model can lose to the simpler one.** Read your `ship_fused` straight off this: green above the dashed line -> maybe ship; green below -> don't.

*Right — the missing-modality drill (no free default).* Each bar is the SAME fused model evaluated when 30% of patients have no image, handled three ways. The dashed line is the no-missing fused score. **drop** trains on fewer, possibly non-representative patients (selection bias — who got imaged?). **zero_impute** tells the head "the X-ray said exactly zero," a confident *lie* that looks like real signal. **flag** adds a column that lets the head learn "trust the image only when it is present." They rarely tie — the gap between bars is the cost of the choice. Read your `missing_policy` off whichever trade-off you can defend, not just the tallest bar.

### Discussion (peer instruction — vote, argue, re-vote)
- Image-only ~0.63, tabular-only ~0.79, fused ~0.77 — **what happened, and would you ship the fused model?** (Name the mechanism, not just "it's worse.")
- The image and tabular data were collected for *different* patients and stitched together. **What assumption does concatenating them quietly make**, and where would that bite a real deployment?
- 30% of patients have no X-ray. You must pick **drop / zero / flag** for a tool that screens an under-resourced clinic where the poorest patients are the *least* likely to have imaging. Which policy, and who does each one quietly harm?

### Stretch (optional)
Swap **late** fusion for **joint** fusion: instead of freezing the encoder, train the encoder + head end-to-end so the image features adapt to the label. Sketch why this needs much more data and overfits faster on 600 patients (you are now fitting the 512-d extractor too, not just a 520-input head). Then re-run the bootstrap CI from the demo cell — does joint fusion's advantage, if any, clear the noise band?